# Study 2 pilot on Gemma 3 27B: can a bigger model tell whether a point is inside a triangle? (preregistered: `docs/preregistration_s2_pilot.md` + `docs/preregistration_s2_pilot_27b.md`)

**Before you press Run all:**
1. Upload the new `geomap_colab.zip` to Google Drive at **`MyDrive/geomap/geomap_colab.zip`** (replace the old one).
2. Colab secrets (key icon), notebook access on: **`HF_TOKEN`** (required), `SLACK_WEBHOOK_URL` (optional).
3. **Runtime → Change runtime type:** pick **H100** or **G4 (RTX PRO 6000)** if the list offers one (the whole model fits,
   fastest). Otherwise pick **A100** with **High-RAM** on: the 27B model (54 GB) does not fit in 40 GB, so part of it
   stays in CPU memory. Same model, same precision, just slower.

Then **Runtime → Run all**. Expected time: about 30–60 minutes on H100/G4, about 1–2 hours on A100 (the first ~10 minutes
are the 54 GB download).

**If Colab disconnects:** reconnect and press **Run all** again. Finished steps are saved in `MyDrive/geomap/results/s2/pilot/`
and skipped on the rerun.


In [ ]:
# 1. GPU and memory check
import subprocess, torch, os
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> A100 / H100 / G4"
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
print(f"GPU {gb:.0f} GB, CPU RAM {ram:.0f} GB")
assert gb > 35, f"Only {gb:.0f} GB of GPU memory: pick A100, H100 or G4"
if gb < 70:
    assert ram > 60, f"Only {ram:.0f} GB of CPU RAM: turn on High-RAM (part of the 27B model has to live in CPU memory)"
    print("Part of the model will stay in CPU memory: slower, same results.")


In [ ]:
# 2. Code from Drive; results live ON Drive (saved as they are produced, kept across disconnects)
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, zipfile
D = "/content/drive/MyDrive/geomap"
BUNDLE, RES = f"{D}/geomap_colab.zip", f"{D}/results"
assert os.path.exists(BUNDLE), f"Upload geomap_colab.zip to {BUNDLE}"
shutil.rmtree("/content/geomap", ignore_errors=True)
zipfile.ZipFile(BUNDLE).extractall("/content/geomap")
if not os.path.exists(RES):
    shutil.copytree("/content/geomap/results", RES)
shutil.rmtree("/content/geomap/results")
os.symlink(RES, "/content/geomap/results")
os.chdir("/content/geomap")
assert os.path.exists("scripts/s2_pilot.py"), "Old bundle: upload the new geomap_colab.zip"
print("already saved:", sorted(os.listdir(f"{RES}/s2/pilot")) if os.path.exists(f"{RES}/s2/pilot") else "nothing yet")

In [ ]:
# 3. Dependencies, PyTorch only (Colab's TensorFlow breaks transformers' Gemma 3 import)
!pip -q install "transformers==4.57.3" nnsight accelerate
import sys
os.environ.update(PYTHONPATH="/content/geomap/src", USE_TF="0", USE_FLAX="0", USE_TORCH="1",
                  TRANSFORMERS_NO_TF="1", TF_CPP_MIN_LOG_LEVEL="3")
sys.path.insert(0, "/content/geomap/src")
!python -c "from transformers.models.gemma3.modeling_gemma3 import Gemma3ForConditionalGeneration; print('Gemma 3 import OK')"

In [ ]:
# 4. Secrets: Hugging Face (required) and Slack (optional)
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import whoami
print("Hugging Face:", whoami(token=os.environ["HF_TOKEN"])["name"])
try:
    os.environ["SLACK_WEBHOOK_URL"] = userdata.get("SLACK_WEBHOOK_URL")
except Exception:
    pass
import sys
sys.modules.pop("geomap", None)  # a stale or wrong "geomap" import must not shadow the code
sys.path.insert(0, "/content/geomap/src")
from geomap.checkpoint import notify
notify("[S2 pilot 27B] Colab run started (or resumed)")

In [ ]:
# 5. Study 2 pilot on 27B: answers on the four-rung ladder (resumes if interrupted)
!cd /content/geomap && PYTHONPATH=/content/geomap/src USE_TF=0 USE_FLAX=0 USE_TORCH=1 TRANSFORMERS_NO_TF=1 python scripts/s2_pilot.py --model gemma3-27b --batch-size 4 2>&1 | grep -v -i warning


In [ ]:
# 6. One zip of the 27B pilot results + the results log, on Drive and downloaded here
import glob
with zipfile.ZipFile("/content/s2_pilot_27b_results.zip", "w") as z:
    for f in glob.glob(f"{RES}/s2/pilot/gemma3-27b_*"):
        z.write(f, "results/s2/pilot/" + os.path.basename(f))
    z.write(f"{RES}/RESULTS.md", "results/RESULTS.md")
shutil.copy("/content/s2_pilot_27b_results.zip", f"{D}/s2_pilot_27b_results.zip")
notify("[S2 pilot 27B] all done: results in MyDrive/geomap/s2_pilot_27b_results.zip")
from google.colab import files
files.download("/content/s2_pilot_27b_results.zip")
